# Decision Trees

Implement a CART classifier from scratch — Gini impurity, best-split search, recursive tree build, and predict — then validate against `sklearn.tree.DecisionTreeClassifier` on Iris data.

## Configuration

Device, seed, and dtype come from `config.toml` via `shared.config.configure()` — never hardcoded. On Apple Silicon this runs on mps.

In [1]:
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # headless-safe under nbconvert
import matplotlib.pyplot as plt  # noqa: E402
import torch  # noqa: E402


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

from shared.config import configure  # noqa: E402

device = configure()
print("running on:", device)

running on: mps


## Imports and data

In [2]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# ── reproducible split ──────────────────────────────────────────────────────
SEED = 42
MAX_DEPTH = 3

iris = load_iris()
X, y = iris.data.astype(np.float64), iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=SEED, stratify=y
)
print(f"Train: {X_train.shape}  Test: {X_test.shape}  Classes: {np.unique(y)}")

Train: (105, 4)  Test: (45, 4)  Classes: [0 1 2]


## CART classifier from scratch

### Gini impurity

$$\text{Gini}(S) = 1 - \sum_k p_k^2$$

A pure node has Gini = 0. The best split maximises the weighted impurity reduction (gain):

$$\text{gain} = \text{Gini}(\text{parent}) - \frac{n_L}{n}\,\text{Gini}(L) - \frac{n_R}{n}\,\text{Gini}(R)$$

### Entropy vs Gini

| Criterion | Formula | Notes |
|-----------|---------|-------|
| **Gini** | $1 - \sum p_k^2$ | Faster; slightly prefers larger partitions |
| **Entropy** | $-\sum p_k \log p_k$ | More information-theoretic; can give different splits |

Both produce similar trees in practice. CART uses Gini by default.

In [3]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Optional

import numpy as np


# ── impurity helpers ─────────────────────────────────────────────────────────

def gini_impurity(labels: np.ndarray) -> float:
    """Gini impurity for a 1-D array of class labels."""
    if labels.size == 0:
        return 0.0
    n = labels.size
    _, counts = np.unique(labels, return_counts=True)
    probs = counts / n
    return float(1.0 - np.sum(probs ** 2))


def weighted_gini(left: np.ndarray, right: np.ndarray) -> float:
    """Weighted child Gini after splitting into left and right."""
    n = left.size + right.size
    if n == 0:
        return 0.0
    return (left.size / n) * gini_impurity(left) + (right.size / n) * gini_impurity(right)


# ── data class for a tree node ───────────────────────────────────────────────

@dataclass
class Node:
    """A node in the CART decision tree."""
    # Leaf attributes
    is_leaf: bool = False
    prediction: Optional[int] = None
    # Internal node attributes
    feature_idx: Optional[int] = None
    threshold: Optional[float] = None
    left: Optional["Node"] = None
    right: Optional["Node"] = None


# ── best-split search ─────────────────────────────────────────────────────────

def _best_split(X: np.ndarray, y: np.ndarray):
    """Search all features and midpoint thresholds; return (feature, threshold, gain)."""
    n_samples, n_features = X.shape
    parent_gini = gini_impurity(y)
    best_gain = -np.inf
    best_feat, best_thresh = None, None

    for feat in range(n_features):
        col = X[:, feat]
        sorted_vals = np.unique(col)
        if sorted_vals.size < 2:
            continue
        # Candidate thresholds: midpoints between consecutive unique values
        thresholds = (sorted_vals[:-1] + sorted_vals[1:]) / 2.0
        for thresh in thresholds:
            mask = col <= thresh
            if mask.sum() == 0 or (~mask).sum() == 0:
                continue
            gain = parent_gini - weighted_gini(y[mask], y[~mask])
            if gain > best_gain:
                best_gain = gain
                best_feat = feat
                best_thresh = thresh

    return best_feat, best_thresh, best_gain


# ── recursive tree builder ────────────────────────────────────────────────────

def _build_tree(X: np.ndarray, y: np.ndarray, depth: int, max_depth: int) -> Node:
    """Recursively build a CART node."""
    # Stopping conditions → leaf
    unique_classes = np.unique(y)
    if (
        unique_classes.size == 1
        or depth >= max_depth
        or X.shape[0] < 2
    ):
        majority = int(np.bincount(y).argmax())
        return Node(is_leaf=True, prediction=majority)

    feat, thresh, gain = _best_split(X, y)
    if feat is None or gain <= 0:
        majority = int(np.bincount(y).argmax())
        return Node(is_leaf=True, prediction=majority)

    mask = X[:, feat] <= thresh
    left = _build_tree(X[mask], y[mask], depth + 1, max_depth)
    right = _build_tree(X[~mask], y[~mask], depth + 1, max_depth)
    return Node(feature_idx=feat, threshold=thresh, left=left, right=right)


# ── predict helpers ───────────────────────────────────────────────────────────

def _predict_one(node: Node, x: np.ndarray) -> int:
    """Walk the tree for one sample."""
    if node.is_leaf:
        return node.prediction  # type: ignore[return-value]
    if x[node.feature_idx] <= node.threshold:  # type: ignore[index]
        return _predict_one(node.left, x)  # type: ignore[arg-type]
    return _predict_one(node.right, x)  # type: ignore[arg-type]


def predict_tree(root: Node, X: np.ndarray) -> np.ndarray:
    """Predict class labels for each row in X."""
    return np.array([_predict_one(root, row) for row in X])


# ── public CART class ─────────────────────────────────────────────────────────

class CARTClassifier:
    """Minimal CART classifier using Gini impurity."""

    def __init__(self, max_depth: int = 3) -> None:
        self.max_depth = max_depth
        self._root: Optional[Node] = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> "CARTClassifier":
        self._root = _build_tree(X, y, depth=0, max_depth=self.max_depth)
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        if self._root is None:
            raise RuntimeError("Call fit() first.")
        return predict_tree(self._root, X)


print("CARTClassifier defined.")

CARTClassifier defined.


## Train the from-scratch CART

In [4]:
cart = CARTClassifier(max_depth=MAX_DEPTH)
cart.fit(X_train, y_train)

preds_scratch = cart.predict(X_test)
acc_scratch = float(np.mean(preds_scratch == y_test))
print(f"From-scratch CART accuracy: {acc_scratch:.4f}")

From-scratch CART accuracy: 0.9333


## Inspect learned splits

In [5]:
def print_tree(node: Node, depth: int = 0, feature_names=None) -> None:
    """Pretty-print the decision tree."""
    indent = "    " * depth
    if node.is_leaf:
        print(f"{indent}Leaf → class {node.prediction}")
        return
    fname = feature_names[node.feature_idx] if feature_names else f"X[{node.feature_idx}]"
    print(f"{indent}if {fname} <= {node.threshold:.4f}:")
    print_tree(node.left, depth + 1, feature_names)
    print(f"{indent}else:")
    print_tree(node.right, depth + 1, feature_names)


print_tree(cart._root, feature_names=iris.feature_names)

if petal length (cm) <= 2.4500:
    Leaf → class 0
else:
    if petal width (cm) <= 1.5500:
        if petal length (cm) <= 4.9500:
            Leaf → class 1
        else:
            Leaf → class 2
    else:
        if petal length (cm) <= 4.8500:
            Leaf → class 1
        else:
            Leaf → class 2


## Validation against sklearn

In [6]:
sk_tree = DecisionTreeClassifier(
    criterion="gini",
    max_depth=MAX_DEPTH,
    random_state=SEED,
)
sk_tree.fit(X_train, y_train)

preds_sklearn = sk_tree.predict(X_test)
acc_sklearn = float(np.mean(preds_sklearn == y_test))
print(f"sklearn  DecisionTreeClassifier accuracy: {acc_sklearn:.4f}")
print(f"from-scratch CART accuracy:               {acc_scratch:.4f}")

TOL = 0.07
diff = abs(acc_scratch - acc_sklearn)
print(f"Difference: {diff:.4f}  (tolerance {TOL})")
assert diff < TOL, (
    f"Accuracy gap too large: scratch={acc_scratch:.4f}, sklearn={acc_sklearn:.4f}, diff={diff:.4f}"
)
print("✓  Assertion passed: from-scratch CART matches sklearn within tolerance.")

sklearn  DecisionTreeClassifier accuracy: 0.9778
from-scratch CART accuracy:               0.9333
Difference: 0.0444  (tolerance 0.07)
✓  Assertion passed: from-scratch CART matches sklearn within tolerance.


## Decision-boundary visualisation (features 2 & 3)

In [7]:
# Use only the first two selected features for 2-D visualisation
feat_a, feat_b = 2, 3  # petal length, petal width
X_2d = X[:, [feat_a, feat_b]]

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_2d, y, test_size=0.3, random_state=SEED, stratify=y
)

cart2 = CARTClassifier(max_depth=MAX_DEPTH)
cart2.fit(X_train2, y_train2)

# Build mesh
h = 0.02
x_min, x_max = X_2d[:, 0].min() - 0.5, X_2d[:, 0].max() + 0.5
y_min, y_max = X_2d[:, 1].min() - 0.5, X_2d[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
mesh_pts = np.c_[xx.ravel(), yy.ravel()]
Z = cart2.predict(mesh_pts).reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]
cmap = plt.cm.RdYlBu

for ax, (clf, title) in zip(
    axes,
    [
        (cart2, "From-scratch CART"),
        (
            DecisionTreeClassifier(criterion="gini", max_depth=MAX_DEPTH, random_state=SEED).fit(X_train2, y_train2),
            "sklearn DecisionTreeClassifier",
        ),
    ],
):
    Z_ = clf.predict(mesh_pts).reshape(xx.shape) if hasattr(clf, "fit") else Z
    ax.contourf(xx, yy, Z_, alpha=0.3, cmap=cmap)
    for cls, color in zip(np.unique(y), colors):
        mask = y_test2 == cls
        ax.scatter(
            X_test2[mask, 0],
            X_test2[mask, 1],
            c=color,
            label=iris.target_names[cls],
            edgecolors="k",
            s=40,
        )
    ax.set_xlabel(iris.feature_names[feat_a])
    ax.set_ylabel(iris.feature_names[feat_b])
    ax.set_title(title)
    ax.legend(loc="upper left")

plt.tight_layout()
plt.savefig("decision_boundary.png", dpi=80, bbox_inches="tight")
plt.show()
print("Plot saved to decision_boundary.png")

Plot saved to decision_boundary.png


/var/folders/gx/cg22rrrs5mx_t0gwgx3809t80000gn/T/ipykernel_70653/196939774.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Gini impurity — worked examples from exercises

In [8]:
# Exercise 1: Gini([6,4]) and Gini([10,0])
def gini_counts(counts):
    n = sum(counts)
    probs = [c / n for c in counts if n > 0]
    return 1 - sum(p**2 for p in probs)

g_64 = gini_counts([6, 4])
g_10 = gini_counts([10, 0])
print(f"Gini([6,4])  = {g_64:.4f}  (expected 0.4800)")
print(f"Gini([10,0]) = {g_10:.4f}  (expected 0.0000)")

assert abs(g_64 - 0.48) < 1e-9
assert abs(g_10 - 0.0) < 1e-9

# Exercise 2: split gain
parent_gini = gini_counts([6, 4])
left_gini   = gini_counts([5, 1])
right_gini  = gini_counts([1, 3])
wgini = (6 / 10) * left_gini + (4 / 10) * right_gini
gain = parent_gini - wgini
print(f"\nSplit gain = {gain:.4f}  (expected ~0.1633)")
assert abs(gain - 0.1633) < 0.001
print("✓  Hand-computed Gini values match expected solutions.")

Gini([6,4])  = 0.4800  (expected 0.4800)
Gini([10,0]) = 0.0000  (expected 0.0000)

Split gain = 0.1633  (expected ~0.1633)
✓  Hand-computed Gini values match expected solutions.


## Takeaways

- **CART** selects the split (feature, threshold) that maximises Gini impurity reduction at each node, recursively.
- **Gini vs Entropy**: both measure node impurity and tend to produce similar trees. Gini is cheaper (no log); entropy is more information-theoretic. CART defaults to Gini.
- **Depth control** is the primary regulariser for a single tree. Too deep → overfit (memorises training labels). Too shallow → high bias.
- **Axis-aligned splits**: a decision tree boundary is always parallel to feature axes — a staircase approximation to any true boundary.
- **Interpretability**: the printed split rules are human-readable, making trees valuable diagnostic tools.
- **Trees as ensemble base learners**: Random Forests and Gradient Boosted Trees average or boost many shallow trees to dramatically reduce variance while retaining low bias.